# 03 — Context dependence

**The core notebook.** Which perturbations behave differently depending on
immune environment?

Two independent readouts, deliberately not one:

1. **Signature divergence** — correlate each perturbation's log2FC signature in
   IFN-γ and co-culture against its signature in control.
2. **E-distance** (`pertpy`, permutation null) — a model-free answer to "did
   this perturbation do anything at all in this condition."

A perturbation is called context-dependent only if **both** hold: a real
effect somewhere, and divergence between environments. Requiring only
divergence would fill the hit list with noise, because two null signatures are
also uncorrelated. That conjunction is the analytical core of the project.

In [ ]:
# =============================================================================
# nb03 — Embedding and structure
#
# Not a clustering notebook. The dominant axis of variation here is condition,
# so clustering cells would recover the experimental design and invite the
# circularity of testing on groups defined by the same expression data.
#
# What the embedding is for: confirming the design held (do the three arms
# separate?), finding structure that was NOT designed and therefore acts as a
# confounder (cell cycle, depth), and checking for T-cell contamination in the
# co-culture arm.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
GUIDE      = s["guide"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]


cc = load_panels()["cell_cycle"]
s_genes   = [g for g in cc["s_phase"]   if g in rna_n.var_names]
g2m_genes = [g for g in cc["g2m_phase"] if g in rna_n.var_names]
print(f"S: {len(s_genes)}/{len(cc['s_phase'])}, "
      f"G2M: {len(g2m_genes)}/{len(cc['g2m_phase'])}")

sc.tl.score_genes_cell_cycle(rna_n, s_genes=s_genes, g2m_genes=g2m_genes)
print(rna_n.obs["phase"].value_counts())


# QC flags from nb02, for annotating anything that turns up here
flags = pd.read_csv(P.tables / "02_gene_flags.csv", index_col=0)
elig  = pd.read_csv(P.tables / "02_guide_eligibility.csv", index_col=0)

print(f"RNA: {rna.n_obs:,} cells x {rna.n_vars:,} genes")
print(f"ADT: {adt.n_obs:,} cells x {adt.n_vars:,} features")
print(f"authors' UMAP present: {'X_umap_orig' in rna.obsm}")
assert set(rna.obs['MOI'].unique()) == {1}

In [ ]:
# Work on a copy — the .h5mu on disk keeps raw counts, and nb04's DE run
# needs those untouched.
# ---- normalise ------------------------------------------------------------
rna_n = rna.copy()
rna_n.X = rna_n.layers["counts"].copy()

sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)

# ---- cell cycle scoring ---------------------------------------------------
# Must happen HERE, on the full gene set. Most cell-cycle genes are not
# variable in control cells and so will not survive HVG selection — scoring
# after the subset would use a fraction of each gene set and give unreliable
# scores. The resulting columns live in .obs and survive the subset.
cc = panels["cell_cycle"]
s_genes   = [g for g in cc["s_phase"]   if g in rna_n.var_names]
g2m_genes = [g for g in cc["g2m_phase"] if g in rna_n.var_names]
print(f"cell cycle genes found — S: {len(s_genes)}/{len(cc['s_phase'])}, "
      f"G2M: {len(g2m_genes)}/{len(cc['g2m_phase'])}")

sc.tl.score_genes_cell_cycle(rna_n, s_genes=s_genes, g2m_genes=g2m_genes)
print(rna_n.obs["phase"].value_counts())

# ---- HVG on control cells only --------------------------------------------
ctrl_cells = rna_n.obs[PERT].astype(str) == CTRL
print(f"control cells: {ctrl_cells.sum():,}")

ctrl_sub = rna_n[ctrl_cells].copy()
sc.pp.highly_variable_genes(ctrl_sub, n_top_genes=cfg["de"]["n_hvg"],
                            flavor="seurat_v3", layer="counts")
rna_n.var["highly_variable"] = rna_n.var_names.isin(
    ctrl_sub.var_names[ctrl_sub.var["highly_variable"]]
)
print(f"HVGs: {rna_n.var['highly_variable'].sum()}")

# ---- scale, PCA, neighbours, UMAP -----------------------------------------
rna_n = rna_n[:, rna_n.var["highly_variable"]].copy()
sc.pp.scale(rna_n, max_value=10)
sc.tl.pca(rna_n, n_comps=50, svd_solver="arpack", random_state=SEED)


In [ ]:
# ---- how many PCs carry signal? -------------------------------------------
sc.pl.pca_variance_ratio(rna_n, n_pcs=50, log=True)

# What loads on the first few PCs. If PC1 is GNLY / CCL5 / NKG7, the dominant
# axis is T-cell contamination rather than biology, and those cells should be
# removed before the embedding means anything.
sc.pl.pca_loadings(rna_n, components=[1, 2, 3])

In [ ]:
# ---- neighbours + UMAP ----------------------------------------------------
N_PCS = 22                       # set from the elbow plot above

sc.pp.neighbors(rna_n, n_neighbors=15, n_pcs=N_PCS, random_state=SEED)
sc.tl.umap(rna_n, random_state=SEED)

# carry the authors' coordinates across for the comparison panel
if "X_umap_orig" in rna.obsm:
    rna_n.obsm["X_umap_orig"] = rna.obsm["X_umap_orig"][
        rna.obs_names.get_indexer(rna_n.obs_names)
    ]

print(rna_n.obsm.keys())

In [ ]:
# ---- quick look -----------------------------------------------------------
# Contamination markers first — this determines whether the embedding is
# usable as-is or needs a cell-removal step.
tcell = [g for g in ["PTPRC", "CD3D", "CD3E", "CD2", "GNLY", "NKG7",
                     "GZMB", "CCL5", "IL32"] if g in rna_n.var_names]
print(f"T-cell markers present in HVG set: {tcell}")

sc.pl.umap(rna_n, color=[COND] + tcell, ncols=3,
           palette=[pal[c] for c in cond_order], frameon=False, s=3)

In [ ]:
# ---- pick a resolution ----------------------------------------------------
# Target is ~3-4 subclusters per condition plus the contaminating T-cell
# island, so roughly 10-13 clusters. Resolution 1.0 gave 27 — too fine for a
# figure, though it was useful for isolating small populations.
# Choose from the composition table, not from how the UMAP looks.
for res in [0.2, 0.3, 0.4, 0.5]:
    key = f"leiden_{res}"
    sc.tl.leiden(rna_n, resolution=res, key_added=key, random_state=SEED,
                 flavor="igraph", n_iterations=2, directed=False)
    n = rna_n.obs[key].nunique()
    # does the T-cell island survive as its own cluster at this resolution?
    tc = rna_n.obs.loc[rna_n.obs["leiden"] == "7", key].value_counts()
    print(f"res {res}: {n:2d} clusters | old cluster 7 lands in "
          f"{tc.index[0]} ({tc.iloc[0]/tc.sum():.0%} of it)")

In [ ]:
RES = 0.3                                    # set from the sweep above
rna_n.obs["cluster"] = rna_n.obs[f"leiden_{RES}"]

print(rna_n.obs["cluster"].value_counts().sort_index())
print()
print(pd.crosstab(rna_n.obs["cluster"], rna_n.obs[COND], normalize="index").round(2))

# flag the T-cell cluster by marker expression rather than by eye
tcell = [g for g in ["PTPRC","CD3D","CD3E","CD2","GNLY","NKG7","GZMB","CCL5","IL32"]
         if g in rna_n.var_names]
tc_score = pd.DataFrame(
    rna_n[:, tcell].X.toarray() if sp.issparse(rna_n.X) else rna_n[:, tcell].X,
    index=rna_n.obs_names, columns=tcell
).mean(axis=1).groupby(rna_n.obs["cluster"]).mean()
print("\nmean T-cell marker score per cluster:")
print(tc_score.sort_values(ascending=False).round(2))

In [ ]:
# ---- which perturbations are non-randomly distributed across clusters? -----
# Confound: clusters are almost perfectly nested within conditions, so raw
# cluster enrichment would mostly recover condition composition. Computed
# WITHIN each condition instead, so the question becomes: given a cell is in
# the IFN-γ arm, does its perturbation predict which IFN-γ subcluster it lands
# in? That is a real question about perturbation-driven state.
MIN_CELLS = 30

rows = []
for cond in cond_order:
    sub = rna_n.obs[rna_n.obs[COND].astype(str) == cond]
    cl_frac = sub["cluster"].value_counts(normalize=True)      # expected
    for pert, d in sub.groupby(PERT, observed=True):
        if len(d) < MIN_CELLS or str(pert) == CTRL:
            continue
        obs = d["cluster"].value_counts(normalize=True)
        for cl in cl_frac.index:
            if cl_frac[cl] < 0.02:            # ignore tiny clusters
                continue
            rows.append({
                "perturbation": str(pert), "condition": cond, "cluster": cl,
                "n": len(d),
                "log2_enrich": np.log2((obs.get(cl, 0) + 0.01) / (cl_frac[cl] + 0.01)),
            })

enrich = pd.DataFrame(rows)
top = (enrich.assign(a=enrich["log2_enrich"].abs())
       .groupby("perturbation")["a"].max()
       .sort_values(ascending=False))
print(top.head(20).round(2))

TOP_N = 12
top_perts = top.head(TOP_N).index.tolist()

In [ ]:
# =============================== FIGURE 3 ==================================
fig, ax = plt.subplot_mosaic(
    """
    AABB
    CCDD
    """,
    figsize=(17, 14),
)

# ---------------------------------- A --------------------------------------
emb = rna_n.obsm["X_umap"]
clusters = rna_n.obs["cluster"].astype(str)
order = sorted(clusters.unique(), key=int)
cmap = plt.cm.tab20(np.linspace(0, 1, len(order)))

for i, cl in enumerate(order):
    m = (clusters == cl).values
    ax["A"].scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.5,
                    color=cmap[i], linewidths=0, rasterized=True)
    ax["A"].text(np.median(emb[m, 0]), np.median(emb[m, 1]), cl,
                 fontsize=10, fontweight="bold", ha="center", va="center",
                 bbox=dict(boxstyle="circle,pad=0.22", facecolor="white",
                           edgecolor="none", alpha=0.75))

ax["A"].set_xticks([]); ax["A"].set_yticks([])
ax["A"].set_xlabel("UMAP 1"); ax["A"].set_ylabel("UMAP 2")
ax["A"].text(0.01, 0.99,
             f"{len(order)} clusters, res = {RES}\n{rna_n.n_obs:,} cells",
             transform=ax["A"].transAxes, ha="left", va="top",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

# ---------------------------------- B --------------------------------------
# Manual dotplot: dot size = fraction of cluster expressing, colour = mean
# scaled expression. Built by hand rather than sc.pl.dotplot because that
# function manages its own figure and does not compose into a mosaic.
#
# These are hand-picked marker sets, NOT genes derived from the clusters —
# testing cluster-defining genes against the clusters that defined them would
# be circular. Asking whether known programs mark known clusters is not.
marker_sets = {
    "T cell":   ["PTPRC", "CD3D", "CD2", "GNLY", "NKG7", "CCL5"],
    "IFN-γ":    ["STAT1", "IRF1", "GBP1", "GBP2", "CXCL10", "SOCS1"],
    "MHC-I":    ["B2M", "HLA-A", "HLA-B", "TAP1", "PSMB9"],
    "inflam.":  ["IL1B", "CXCL8", "CCL2", "MMP1", "S100A8"],
    "cycle":    ["MKI67", "TOP2A", "CCNB2"],
}
genes = [g for gs in marker_sets.values() for g in gs if g in rna_n.var_names]

X = rna_n[:, genes].X
X = np.asarray(X.todense()) if sp.issparse(X) else np.asarray(X)
Xdf = pd.DataFrame(X, index=rna_n.obs_names, columns=genes)
grp = rna_n.obs["cluster"].astype(str).values

frac = Xdf.gt(0).groupby(grp).mean()                      # fraction expressing
mean = Xdf.groupby(grp).mean()
mean = (mean - mean.min()) / (mean.max() - mean.min())    # scale per gene
frac, mean = frac.loc[order], mean.loc[order]

gx, gy = np.meshgrid(np.arange(len(genes)), np.arange(len(order)))
sc_ = ax["B"].scatter(gx.ravel(), gy.ravel(),
                      s=frac.values.ravel() * 130,
                      c=mean.values.ravel(), cmap="Blues",
                      edgecolor="grey", linewidth=0.2, vmin=0, vmax=1)

ax["B"].set_xticks(range(len(genes)))
ax["B"].set_xticklabels(genes, rotation=90, fontsize=7)
ax["B"].set_yticks(range(len(order)))
ax["B"].set_yticklabels(order, fontsize=8)
ax["B"].set_ylabel("cluster")
ax["B"].invert_yaxis()
ax["B"].set_xlim(-0.7, len(genes) - 0.3)
ax["B"].set_ylim(len(order) - 0.4, -2.0)
ax["B"].grid(alpha=0.15)

# set boundaries
b = 0
for name, gs in marker_sets.items():
    n = len([g for g in gs if g in genes])
    if n == 0:
        continue
    ax["B"].axvline(b - 0.5, color="k", lw=0.6)
    ax["B"].text(b + n / 2 - 0.5, -1.4, name, ha="center", fontsize=7,
             fontweight="bold")
    b += n
plt.colorbar(sc_, ax=ax["B"], label="scaled mean expr.", shrink=0.6)

# ---------------------------------- C --------------------------------------
for cond in cond_order:
    m = (rna_n.obs[COND].astype(str) == cond).values
    ax["C"].scatter(emb[m, 0], emb[m, 1], s=1.2, alpha=0.5,
                    color=pal.get(cond, "#888"), linewidths=0,
                    label=f"{cond} (n={m.sum():,})", rasterized=True)

ax["C"].set_xticks([]); ax["C"].set_yticks([])
ax["C"].set_xlabel("UMAP 1"); ax["C"].set_ylabel("UMAP 2")
ax["C"].legend(fontsize=9, loc="upper left", markerscale=8, framealpha=0.9)

# ---------------------------------- D --------------------------------------
# The perturbations whose cells are most unevenly distributed across
# subclusters WITHIN their own condition. Everything else in grey.
ax["D"].scatter(emb[:, 0], emb[:, 1], s=0.8, color="#dddddd",
                linewidths=0, rasterized=True)

dcmap = plt.cm.tab20(np.linspace(0, 1, len(top_perts)))
pert_col = rna_n.obs[PERT].astype(str).values
for i, p in enumerate(top_perts):
    m = pert_col == p
    ax["D"].scatter(emb[m, 0], emb[m, 1], s=3.5, alpha=0.85,
                    color=dcmap[i], linewidths=0,
                    label=f"{p} (n={m.sum():,})", rasterized=True)

ax["D"].set_xticks([]); ax["D"].set_yticks([])
ax["D"].set_xlabel("UMAP 1"); ax["D"].set_ylabel("UMAP 2")
ax["D"].legend(fontsize=7, loc="upper left", markerscale=4, ncol=2,
               framealpha=0.9)

titles = {
    "A": f"Leiden clusters (res {RES})",
    "B": "marker expression by cluster",
    "C": "condition",
    "D": f"top {TOP_N} perturbations by within-condition cluster enrichment",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "03_embedding", cfg)